# First-approach component MD — each pair's movement up to the moment it first gets there

`component_md.ipynb` measures component MD over the pair's whole movement: from the pair
leaving baseline to the start of the hold that succeeded. In a 4-finger chord that window
often runs on long after the pair first reached its target: a finger drifts out of its zone
while the other pair is still being worked on, then comes back. Those later adjustments
count towards the pair's "synchrony" and hide what the pair did on its first go.

**First-approach component MD** keeps the same start and the same smoothed 2-D path, but
ends the window at the pair's **first approach**: the onset of the first period in which
both fingers are in their target zones and that

* lasts at least `HOLD_MS` (100 ms), or
* is the final hold itself.

The hold requirement skips passing through a zone on the way to overshooting it (the
extension and flexion zones are bounded, 2 to 5 and −5 to −2). The first approach is never
later than the normal window end; when the pair got there in one go the two windows are
identical. Detection is `utils/events.py: first_approach`, on the unsmoothed force with the
task's own zone rule, like the other events.

Written into each subject's `analysis/s<sn>_component_md.csv`, right after `cMD`:

| column | meaning |
|---|---|
| `fa_end` | first approach, ms from the go cue |
| `fa_dur` | `fa_end - pair_start`, the first-approach window length |
| `cMD_fa` | component MD over `pair_start` → `fa_end` |

**Run order:** `preprocess.ipynb` → `component_md.ipynb` (writes the csv) → this notebook.
Re-running `component_md.ipynb` rewrites the csv without these columns, so re-run this one
after it.

No figures here; the tables at the end compare `cMD_fa` with `cMD`.

## Imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "target").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.design import TEST1_BLOCKS, TEST2_BLOCKS
from utils.events import WAIT_EXEC, first_approach

SUBJECTS = [100, 101, 102, 103, 104]   # <<< every subject with an s<sn>_component_md.csv
HOLD_MS = 100                          # <<< time in the zones that counts as having arrived

ANALYSIS_DIR = ROOT / "analysis"
SLOT_FINGERS = {"thumb+index": (0, 1), "ring+pinky": (3, 4)}
FA_COLS = ["fa_end", "fa_dur", "cMD_fa"]

## First approach and its component MD for every (trial, slot)

`mean_deviation` and `path` are the ones in `component_md.ipynb`. Two checks per subject:

1. recomputing component MD over the stored window (`pair_start` → `pair_end`) must give the
   stored `cMD`, so this notebook and `component_md.ipynb` cannot drift apart;
2. `pair_start ≤ fa_end ≤ pair_end` on every row, and `fa_end` is missing exactly where
   `pair_end` is.

In [2]:
def mean_deviation(F):
    """Mean distance of a force path from the straight line joining its ends.

    F is (n_samples, n_dims). Returns NaN when the path is too short or does not move.
    """
    if len(F) < 3:
        return np.nan
    c = F[-1] - F[0]
    if c @ c == 0:
        return np.nan
    d = F - F[0]
    return np.linalg.norm(d - np.outer(d @ c / (c @ c), c), axis=1)[1:].mean()


def path(rec, start, end, fingers):
    """Smoothed force of `fingers` from start to end (ms from the go cue), inclusive."""
    t0 = rec.t[rec.state == WAIT_EXEC][0]
    return rec.force[(rec.t >= t0 + start) & (rec.t <= t0 + end)][:, list(fingers)]


for sn in SUBJECTS:
    df = pd.read_csv(ANALYSIS_DIR / f"s{sn}.csv")
    mov = pd.read_pickle(ANALYSIS_DIR / f"s{sn}_mov.pkl")
    cmd = pd.read_csv(ANALYSIS_DIR / f"s{sn}_component_md.csv").drop(columns=FA_COLS, errors="ignore")
    REC = {(r.BN, r.TN): r for r in mov.itertuples()}
    TRIAL = {(r.BN, r.TN): r for r in df.itertuples()}
    assert set(cmd.slot) <= set(SLOT_FINGERS), f"s{sn}: unexpected slot names {set(cmd.slot)}"

    fa_end, cmd_fa, cmd_check = [], [], []
    for r in cmd.itertuples():
        rec, tr, fingers = REC[(r.BN, r.TN)], TRIAL[(r.BN, r.TN)], SLOT_FINGERS[r.slot]
        e = first_approach(rec, tr, fingers, r.pair_start, r.pair_end, HOLD_MS)
        fa_end.append(e)
        cmd_fa.append(mean_deviation(path(rec, r.pair_start, e, fingers)) if np.isfinite(e) else np.nan)
        cmd_check.append(mean_deviation(path(rec, r.pair_start, r.pair_end, fingers))
                         if np.isfinite(r.pair_end) else np.nan)

    cmd["fa_end"] = fa_end
    cmd["fa_dur"] = cmd.fa_end - cmd.pair_start
    cmd["cMD_fa"] = cmd_fa

    assert np.allclose(cmd_check, cmd.cMD, equal_nan=True, atol=1e-9), \
        f"s{sn}: component MD over the stored window != stored cMD - re-run component_md.ipynb"
    found = cmd.fa_end.notna()
    assert found.equals(cmd.pair_end.notna()), f"s{sn}: fa_end missing where pair_end is not"
    assert (cmd.fa_end[found] >= cmd.pair_start[found]).all() \
        and (cmd.fa_end[found] <= cmd.pair_end[found]).all(), f"s{sn}: first approach outside the window"

    cols = list(cmd.columns.drop(FA_COLS))
    i = cols.index("cMD") + 1
    cmd = cmd[cols[:i] + FA_COLS + cols[i:]]
    cmd.to_csv(ANALYSIS_DIR / f"s{sn}_component_md.csv", index=False)

    earlier = (cmd.fa_end < cmd.pair_end)[found].mean()
    print(f"s{sn}: {len(cmd)} (trial, slot) rows; first approach ends the window earlier on "
          f"{earlier:.0%}; {int(cmd.cMD_fa.isna().sum() - (~found).sum())} window(s) too short for MD "
          f"-> analysis/s{sn}_component_md.csv")

s100: 842 (trial, slot) rows; first approach ends the window earlier on 24%; 0 window(s) too short for MD -> analysis/s100_component_md.csv
s101: 1447 (trial, slot) rows; first approach ends the window earlier on 23%; 0 window(s) too short for MD -> analysis/s101_component_md.csv


s102: 976 (trial, slot) rows; first approach ends the window earlier on 22%; 0 window(s) too short for MD -> analysis/s102_component_md.csv
s103: 1269 (trial, slot) rows; first approach ends the window earlier on 19%; 0 window(s) too short for MD -> analysis/s103_component_md.csv


s104: 1295 (trial, slot) rows; first approach ends the window earlier on 35%; 0 window(s) too short for MD -> analysis/s104_component_md.csv


## Summary: first-approach vs normal component MD

Per subject and condition, over correct trials: how often the first approach ends the
window earlier than the normal end, and the medians of both measures and both window
lengths (ms). "alone" = the 2-finger component in training; in the 4-finger blocks each
slot is either the trained component or the flipped, untrained pattern.

In [3]:
cmd_all = pd.concat([pd.read_csv(ANALYSIS_DIR / f"s{sn}_component_md.csv") for sn in SUBJECTS],
                    ignore_index=True)
cmd_all["condition"] = np.where(cmd_all.kind == "2f", "alone",
                                np.where(cmd_all.trained, "trained slot", "flipped slot"))
cmd_all["fa_earlier"] = cmd_all.fa_end < cmd_all.pair_end

summary = (cmd_all.groupby(["slot", "condition", "sn"])
           .agg(n=("cMD", "size"), fa_earlier=("fa_earlier", "mean"),
                cMD=("cMD", "median"), cMD_fa=("cMD_fa", "median"),
                pair_dur=("pair_dur", "median"), fa_dur=("fa_dur", "median")))
print(summary.round(3).to_string())

print("\nGroup: mean of the per-subject medians")
print(summary.groupby(["slot", "condition"])[["fa_earlier", "cMD", "cMD_fa", "pair_dur", "fa_dur"]]
      .mean().round(3).to_string())

                                n  fa_earlier    cMD  cMD_fa  pair_dur  fa_dur
slot        condition    sn                                                   
ring+pinky  alone        100  293       0.205  0.526   0.519     472.0   414.0
                         101  297       0.246  0.200   0.149     254.0   176.0
                         102   98       0.276  0.311   0.290     865.0   756.0
                         103  198       0.066  0.820   0.819     551.0   541.0
                         104  291       0.162  0.369   0.354     348.0   322.0
            flipped slot 100   60       0.500  0.699   0.731    1890.0   724.0
                         101  208       0.399  0.633   0.570     799.0   578.0
                         102  185       0.405  0.737   0.836    1116.0   834.0
                         103  205       0.385  0.950   0.963     928.0   736.0
                         104  171       0.661  0.618   0.646    2400.0  1192.0
            trained slot 100   66       0.409  0.816

## Test 1 → Test 2, trained vs flipped slot

The comparison this measure was introduced for. Per subject and slot: the median in Test 1
and Test 2 with the slot as the trained component and as the flipped pattern, and

**transfer** = (Test 1 − Test 2) for the trained slot − (Test 1 − Test 2) for the flipped slot.

Positive = the trained pattern got more synchronous after component training than the
untrained one. Correct trials only, so a subject with few correct Test 1 trials of a slot
rests on few trials (the `n` table), and one with none gets NaN.

In [4]:
tests = cmd_all[cmd_all.kind == "4f"].assign(
    test=lambda d: np.select([d.BN.isin(TEST1_BLOCKS), d.BN.isin(TEST2_BLOCKS)], ["Test 1", "Test 2"], ""))
tests = tests[tests.test != ""]

print("n correct trials")
print(tests.groupby(["slot", "sn", "condition", "test"]).size()
      .unstack(["condition", "test"]).fillna(0).astype(int).to_string())

for col in ["cMD", "cMD_fa"]:
    m = tests.groupby(["slot", "sn", "condition", "test"])[col].median().unstack(["condition", "test"])
    m[("transfer", "")] = ((m[("trained slot", "Test 1")] - m[("trained slot", "Test 2")])
                           - (m[("flipped slot", "Test 1")] - m[("flipped slot", "Test 2")]))
    print(f"\n{col}  ({'normal window' if col == 'cMD' else 'first-approach window'})")
    print(m.round(3).to_string())
    print("group mean transfer:", m[("transfer", "")].groupby(level="slot").mean().round(3).to_dict())

n correct trials
condition       flipped slot        trained slot       
test                  Test 1 Test 2       Test 1 Test 2
slot        sn                                         
ring+pinky  100           22     38           28     38
            101           30     35           39     44
            102           14     35            0     10
            103           25     40            1     38
            104           21     26           14     35
thumb+index 100           25     40           25     36
            101           31     43           38     36
            102            6     27            8     18
            103           13     32           13     46
            104           16     23           19     38

cMD  (normal window)
condition       flipped slot        trained slot        transfer
test                  Test 1 Test 2       Test 1 Test 2         
slot        sn                                                  
ring+pinky  100        0.716  0.686   